# Alignment of multigraph snapshots (Poisson Model)

We perform alignment of snapshots of a multigraph using the Poisson model. We run Monte Carlo for several runs to obtain a bigger ensemble of alignments, from which we will finally infer both the mapping of the nodes and the underlying blueprint (see notebook: analysis_alignment.ipynb).

**Note:** The adjacency matrices must contain integer counts, as the Poisson model requires discrete edge values. 
So in that case, we would need to transform it first with `A = np.rint(A).astype(int)`.

In this example, we use reduced versions of the original networks to provide a quick demonstration.

In [1]:
#Our package:
from aligner.networksHandling import helper as helper_networks
from aligner.poissonAlignment import helper as helper_poisson
from aligner.montecarlo import helper as helper_montecarlo

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle 

In [3]:
#create directories to store results (if they don't exist)
!mkdir -p ../results/
!mkdir -p ../results/celegans/

## 1. Alignment without groups

We want to align the snapshots of the network stored in `A` of size $T\times N$. T is the number of snapshots and N is the number of nodes.

In here, we will work with the last four C. elegans connectomes (reduced version).

In [4]:
labels = ["L2", "L3", "A.1", "A.2"]

A = helper_networks.load_networks(data_folder="../data/processed/celegans/", 
                                  net_ids=[4,5,6,7], 
                                  file_prefix="reduced_")
T,N,N = A.shape
print(f"T (number of snapshots) = {T}\nN (number of nodes) = {N}")

T (number of snapshots) = 4
N (number of nodes) = 60


In [5]:
#compute the alpha of each snapshot (developmental parameter)
ST = helper_poisson.get_ST(A)
alphas = helper_poisson.get_alpha(A, ST)
print(f"Developmental parameter per snapshot: {alphas}")

Developmental parameter per snapshot: [0.54421222 0.54662379 0.88665595 1.        ]


In [6]:
#Before alinging the connectomes, let's compute the GROUND TRUTH energy:
gtP = np.array([np.arange(N) for t in range(T)]) #GT permutation is the same permutation for each snapshot (bc nodes are originally ordered in the same way)

#Poisson parameters needed:
Salpha = helper_poisson.get_Salpha(A) #sum over alphas (developmental parameters)

Egt = helper_poisson.calculateEnergy_total(A, gtP, Salpha, N, T)
print(f"Ground Truth Energy = {Egt}")

Ground Truth Energy = -6382.846891649653


In [7]:
#Parameters of the Monte Carlo:
## Parallel tempering:
beta_spacing = 1.005
betas = np.array([beta_spacing**n for n in range(-12,1)])
## Runs:
R = 5
MCsteps = 1000
start_sampling = 500
freq_sampling = 2

#Initial permutation of the snapshots:
initial_P = helper_montecarlo.initial_permutation(A) #nodes are ordered by strength

In [8]:
#Perform the alignment and store both internal and final results:

ensemble = [] #in here, we'll store all the samples from the different runs

for run in range(R):  
    
    Esweeps_r, samples, groundState, MCacceptance_rate = helper_montecarlo.sampling_poisson(A=A, 
        P=initial_P, betas=betas, S=MCsteps, sampleFrom=start_sampling, sampleFreq=freq_sampling, 
        filesave="../results/celegans/internal_results_nogroups")
        
    final_results = dict({"E": Esweeps_r, "E_GT": Egt, "GS": groundState, "Samples": samples})
    with open(f"../results/celegans/nogroups_run{run}.pkl", "wb") as file:
        pickle.dump(final_results, file)

    ensemble.append(samples)

Sweeps: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:33<00:00, 30.14it/s]


Data saved for sweep 1000.
Replica with minimum energy: 11


Sweeps: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:30<00:00, 32.50it/s]


Data saved for sweep 1000.
Replica with minimum energy: 11


Sweeps: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:30<00:00, 32.39it/s]


Data saved for sweep 1000.
Replica with minimum energy: 11


Sweeps: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:30<00:00, 32.60it/s]


Data saved for sweep 1000.
Replica with minimum energy: 11


Sweeps: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:30<00:00, 32.45it/s]

Data saved for sweep 1000.
Replica with minimum energy: 10


In [9]:
#to see last permutation sampled from the last run:
ensemble[-1][-1]

array([[13, 12, 11, 10, 28,  6, 21, 44,  0, 29, 18,  7, 20, 39, 41, 36,
        38, 45, 31, 15, 32, 27, 47, 14,  3, 56,  2, 19, 49, 55, 58, 48,
        33, 35, 46, 59, 42,  9, 40,  4,  8,  5, 52,  1, 51, 37, 30, 53,
        54, 22, 34, 17, 50, 57, 23, 24, 16, 25, 26, 43],
       [13, 12, 11, 10, 28,  6, 21, 44,  0, 29, 19,  7, 20, 39, 41, 36,
        38, 45, 31, 15, 32, 27, 47, 14,  3, 57,  2, 18, 49, 55, 26, 48,
        33, 35, 46, 59, 42,  8, 40,  4,  9,  5, 25,  1, 50, 37, 30, 53,
        54, 56, 34, 17, 51, 22, 23, 24, 16, 58, 52, 43],
       [12, 13, 10, 11, 29,  7, 20, 45,  1, 28, 19,  6, 21, 38, 40, 37,
        39, 44, 30, 14, 33, 27, 46, 15,  2, 57,  3, 18, 48, 54, 58, 49,
        32, 34, 47, 59, 43,  8, 41,  5,  9,  4, 53,  0, 51, 36, 31, 52,
        55, 56, 35, 16, 50, 22, 24, 23, 17, 26, 25, 42],
       [13, 12, 11, 10, 28,  6, 21, 44,  0, 29, 19,  7, 20, 39, 41, 36,
        38, 45, 31, 15, 32, 27, 47, 14,  3, 56,  2, 18, 49, 55, 26, 48,
        33, 35, 46, 59, 42,  8, 40,  

## 2. Group-guided alignment (e.g. C. elegans connectomes constraining the permutations by the location of the neurons: left vs right)

Besides the adjacency matrices, we also need the array:

- `groups`: indicating the group membership of each node (nodes are ordered in the same way as in the adjacency matrices A).

In [10]:
labels = ["L2", "L3", "A.1", "A.2"]

A = helper_networks.load_networks(data_folder="../data/processed/celegans/", 
                                  net_ids=[4,5,6,7], 
                                  file_prefix="reduced_")
T,N,N = A.shape
print(f"T (number of snapshots) = {T}\nN (number of nodes) = {N}")

T (number of snapshots) = 4
N (number of nodes) = 60


In [11]:
#compute the alpha of each snapshot (developmental parameter)
ST = helper_poisson.get_ST(A)
alphas = helper_poisson.get_alpha(A, ST)
print(f"Developmental parameter per snapshot: {alphas}")

Developmental parameter per snapshot: [0.54421222 0.54662379 0.88665595 1.        ]


In [12]:
#Before alinging the connectomes, let's compute the GROUND TRUTH energy:
gtP = np.array([np.arange(N) for t in range(T)]) #GT permutation is the same permutation for each snapshot (bc nodes are originally ordered in the same way)

#Poisson parameters needed:
Salpha = helper_poisson.get_Salpha(A) #sum over alphas

Egt = helper_poisson.calculateEnergy_total(A, gtP, Salpha, N, T)
print(f"Ground Truth Energy = {Egt}")

Ground Truth Energy = -6382.846891649653


In [13]:
neuron_metadata = pd.read_csv("../data/processed/celegans/reduced_neuron_metadata.csv", index_col=0) #neuron order in adj mats
neuron_metadata.head()

,name,type,location
0,ADAL,Inter,Left
1,ADAR,Inter,Right
2,ADEL,Modulatory,Left
3,ADER,Modulatory,Right
4,ADFL,Sensory,Left


In [14]:
groups = np.array(neuron_metadata["location"])

In [15]:
#The initial permutation of the snapshots changes (we want to follow the group structure of the nodes)
initial_P_groups = helper_montecarlo.initial_permutation_groups(A, groups) 

Groups: ['Left' 'Other' 'Right']
Group boundary array: [ 0 29 31 60]


In [16]:
#Perform the GROUP-GUIDED alignment and store both internal and final results (parameters are the same as before, as well as the GT Energy):

ensemble = [] #in here, we'll store all the samples from the different runs

for run in range(R):  
    
    Esweeps_r, samples, groundState, MCacceptance_rate = helper_montecarlo.sampling_poisson_groups(A=A, 
        P=initial_P_groups, betas=betas, S=MCsteps, sampleFrom=start_sampling, sampleFreq=freq_sampling, 
        groups=groups, filesave="../results/celegans/groups_internal_results")
        
    final_results = dict({"E": Esweeps_r, "E_GT": Egt, "GS": groundState, "Samples": samples})
    with open(f"../results/celegans/groups_run{run}.pkl", "wb") as file:
        pickle.dump(final_results, file)

    ensemble.append(samples)

Sweeps: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:30<00:00, 33.06it/s]


Data saved for sweep 1000.
Replica with minimum energy: 7


Sweeps: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:29<00:00, 33.57it/s]


Data saved for sweep 1000.
Replica with minimum energy: 8


Sweeps: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:29<00:00, 33.59it/s]


Data saved for sweep 1000.
Replica with minimum energy: 11


Sweeps: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:29<00:00, 33.63it/s]


Data saved for sweep 1000.
Replica with minimum energy: 10


Sweeps: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:29<00:00, 33.62it/s]

Data saved for sweep 1000.
Replica with minimum energy: 7


In [17]:
#to see last permutation sampled from the last run:
ensemble[-1][-1]

array([[12, 10, 20,  6, 32, 28, 40, 18, 30,  0, 42,  4, 48, 54, 25,  8,
        52, 58, 44, 34, 16, 38, 14, 23, 56, 46, 50,  2, 36, 59, 22, 13,
        11, 21, 33, 39, 45, 29, 15,  7, 49, 37, 35, 53, 43, 47, 24, 27,
        41,  9, 17, 55, 19, 31,  3, 57,  1, 26,  5, 51],
       [12, 10, 20,  6, 32, 28, 40, 18, 30,  0, 42,  4, 48, 54, 25,  8,
        52, 23, 44, 34, 16, 38, 14, 58, 56, 46, 50,  2, 36, 59, 22, 13,
        11, 21, 33, 39, 45, 29, 15,  7, 49, 37, 35, 53, 43, 47, 24, 27,
        41,  9, 17, 55, 19, 31,  3, 57,  1, 26,  5, 51],
       [12, 10, 20,  6, 32, 28, 40, 18, 30,  0, 42,  4, 48, 54, 23,  8,
        52, 58, 44, 34, 16, 38, 14, 25, 56, 46, 50,  2, 36, 59, 22, 13,
        11, 21, 33, 39, 45, 29, 15,  7, 49, 37, 35, 53, 43, 47, 24, 27,
        41,  9, 17, 55, 19, 31,  3, 57,  1, 26,  5, 51],
       [12, 10, 20,  6, 32, 28, 40, 18, 30,  0, 42,  4, 48, 54, 23,  8,
        52, 58, 44, 34, 16, 38, 14, 25, 56, 46, 50,  2, 36, 59, 22, 13,
        11, 21, 33, 39, 45, 29, 15,  